# Imports and Configs

In [1]:
import os
from src.config import settings

import pandas as pd
import numpy as np

from src.data.loader import RetailRocketLoader
from src.data.preprocessor import EventWeightPreprocessor, MinInteractionsFilter
from src.features.engineering import encode_ids, split_interactions

In [2]:
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
settings.mlflow_tracking_uri = "file:./mlruns"

In [3]:
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print(f"Current working directory: {os.getcwd()}")

Current working directory: c:\Users\giova\Documents\FIAP - MLE\2 - Big Data Architecture\retailrocket-recsys


# Load Data

In [4]:
loader = RetailRocketLoader()
events_df = loader.load_events()
items_df = loader.load_item_properties()

INFO: RetailRocketLoader initialized with path: data\raw
INFO: Loading events from: data\raw\events.csv
INFO: Raw events loaded: 2756101 rows
INFO: Events validated: 2756101 → 2756101 rows | distribution: {'view': 2664312, 'addtocart': 69332, 'transaction': 22457}
INFO: Loading item properties from 2 files
INFO: Item properties loaded: 20275902 rows, 417053 unique items


# Preprocessing

In [5]:
# 1. Instanciar as classes de pré-processamento
weight_preprocessor = EventWeightPreprocessor()
interactions_filter = MinInteractionsFilter(min_user=5, min_item=5)

# 2. Aplicar o mapeamento de pesos e agrupamento (soma dos scores por user-item)
print("Aplicando mapeamento de pesos...")
events_weighted = weight_preprocessor.transform(events_df)

# 3. Aplicar a filtragem de interações mínimas (5)
print("\nFiltrando interações mínimas...")
events_filtered = interactions_filter.transform(events_weighted)

# Visualizar as primeiras linhas do resultado filtrado
events_filtered.head()

INFO: MinInteractionsFilter initialized: min_user=5, min_item=5
INFO: EventWeightPreprocessor: applying weights {'view': 1, 'addtocart': 3, 'transaction': 5} to 2756101 events


Aplicando mapeamento de pesos...


INFO: EventWeightPreprocessor: aggregated to 2145179 (user, item) pairs | score range: [1.0, 308.0]



Filtrando interações mínimas...


INFO: MinInteractionsFilter: 2145179 → 312680 rows | users: 1407580 → 38178 | items: 235061 → 23188


,visitorid,itemid,score
0,51,49967,1
1,51,198762,2
2,51,429304,1
3,54,38965,2
4,54,249114,1


# IDs Encoding & Splits

In [6]:
# 1. Executar o mapeamento de IDs (Categorificação)
encoded_data = encode_ids(events_filtered)

print("Mapeamento concluído:")
print(f"  Número de usuários únicos: {encoded_data.num_users}")
print(f"  Número de itens únicos: {encoded_data.num_items}")

# 2. Dividir os dados em treino, validação e teste
splits = split_interactions(encoded_data.df, val_size=0.1, test_size=0.1)

print("\nDivisão dos dados concluída:")
print(f"  Treino:    {len(splits.train)} interações")
print(f"  Validação: {len(splits.val)} interações")
print(f"  Teste:     {len(splits.test)} interações")

Mapeamento concluído:
  Número de usuários únicos: 38178
  Número de itens únicos: 23188

Divisão dos dados concluída:
  Treino:    250143 interações
  Validação: 31269 interações
  Teste:     31268 interações


# Popularity Baselines Calculation

In [7]:
# Calculamos a popularidade dos itens a partir do conjunto de TREINO

# 1. Popularidade por Soma de Scores (pesos atribuídos a cada evento: view=1, addtocart=3, transaction=5)
item_popularity_score = splits.train.groupby("item_idx")["score"].sum().sort_values(ascending=False)
popular_items_by_score = item_popularity_score.index.tolist()

# 2. Popularidade por Contagem de Eventos (total de interações independentemente do peso)
item_popularity_count = splits.train.groupby("item_idx").size().sort_values(ascending=False)
popular_items_by_count = item_popularity_count.index.tolist()

print(f"Total de itens populares mapeados: {len(popular_items_by_score)}")
print("Top 5 itens mais populares (por Score):")
print(item_popularity_score.head(5))
print("\nTop 5 itens mais populares (por Contagem):")
print(item_popularity_count.head(5))

Total de itens populares mapeados: 23188
Top 5 itens mais populares (por Score):
item_idx
22935    1062
5936     1007
476       801
15576     674
12038     584
Name: score, dtype: int64

Top 5 itens mais populares (por Contagem):
item_idx
12796    254
11679    218
15455    208
22935    208
476      195
dtype: int64


# Evaluation Setup

Aqui configuramos os parâmetros de avaliação de ranking. 

> [!NOTE]
> Por padrão, definimos `_EVAL_USERS = 200` para manter consistência direta com a Fatoração de Matrizes. 
> No entanto, devido à esparsidade dos dados, a avaliação em apenas 200 usuários pode resultar em métricas zeradas (`0.0`).
> Para avaliar na base de testes completa, altere a variável abaixo para `_EVAL_USERS = len(ground_truth)`.

In [13]:
# Mapeamento de itens vistos no treino
seen_in_train = splits.train.groupby("user_idx")["item_idx"].apply(set).to_dict()

# Gabarito do conjunto de teste
ground_truth = splits.test.groupby("user_idx")["item_idx"].apply(set).to_dict()

# Parâmetros de ranking
k = 10

# Para avaliação rápida comparativa (padrão): 200. Para base cheia: len(ground_truth)
_EVAL_USERS = len(ground_truth)

# Seleção da amostra de usuários de teste
test_users = list(ground_truth.keys())[:_EVAL_USERS]
print(f"Avaliando ranking Top-{k} em {len(test_users)} usuários de teste.")

Avaliando ranking Top-10 em 17237 usuários de teste.


# Evaluation - Scenario A (Fair Comparison: Filtered Seen Items)

Neste cenário, filtramos os itens que cada usuário já interagiu no conjunto de treino. Isso replica exatamente o comportamento de avaliação do modelo de Fatoração de Matrizes.

In [14]:
from src.evaluation.metrics import compute_all_metrics

# --- 1. Avaliação do Baseline por Soma de Scores ---
recs_by_score = {}
for uid in test_users:
    user_seen = seen_in_train.get(uid, set())
    rec_list = []
    for item in popular_items_by_score:
        if item not in user_seen:
            rec_list.append(item)
            if len(rec_list) == k:
                break
    recs_by_score[uid] = rec_list

metrics_a_score_df = compute_all_metrics(recs_by_score, ground_truth, k=k)

# --- 2. Avaliação do Baseline por Contagem de Interações ---
recs_by_count = {}
for uid in test_users:
    user_seen = seen_in_train.get(uid, set())
    rec_list = []
    for item in popular_items_by_count:
        if item not in user_seen:
            rec_list.append(item)
            if len(rec_list) == k:
                break
    recs_by_count[uid] = rec_list

metrics_a_count_df = compute_all_metrics(recs_by_count, ground_truth, k=k)

print("=== Métricas Cenário A (Filtrado por Itens Vistos no Treino) calculadas! ===")

=== Métricas Cenário A (Filtrado por Itens Vistos no Treino) calculadas! ===


# Evaluation - Scenario B (Standard Popularity: Unfiltered Seen Items)

Neste cenário, recomendamos os itens mais populares absolutos, mesmo se o usuário já tiver interagido com eles no treino.

In [15]:
# --- 1. Avaliação do Baseline por Soma de Scores (Sem filtro) ---
recs_by_score_unfiltered = {}
top_k_score_popular = popular_items_by_score[:k]
for uid in test_users:
    recs_by_score_unfiltered[uid] = top_k_score_popular

metrics_b_score_df = compute_all_metrics(recs_by_score_unfiltered, ground_truth, k=k)

# --- 2. Avaliação do Baseline por Contagem de Interações (Sem filtro) ---
recs_by_count_unfiltered = {}
top_k_count_popular = popular_items_by_count[:k]
for uid in test_users:
    recs_by_count_unfiltered[uid] = top_k_count_popular

metrics_b_count_df = compute_all_metrics(recs_by_count_unfiltered, ground_truth, k=k)

print("=== Métricas Cenário B (Sem Filtrar Itens Vistos no Treino) calculadas! ===")

=== Métricas Cenário B (Sem Filtrar Itens Vistos no Treino) calculadas! ===


# Consolidated Comparison Table

Abaixo geramos a tabela comparativa programática contendo os baselines de popularidade calculados e os resultados da Fatoração de Matrizes (carregados manualmente para a comparação).

In [16]:
# Configurar precisão de exibição do Pandas para ver casas decimais pequenas
pd.options.display.float_format = '{:.6f}'.format

# Dicionário com os resultados da Fatoração de Matrizes (obtido em matrix_factorization.ipynb com 200 usuários)
mf_metrics = {
    f"precision@{k}": 0.002500,
    f"recall@{k}": 0.018750,
    f"ndcg@{k}": 0.010361,
    f"hit_rate@{k}": 0.025000
}

# Criando DataFrame programático unificado
comparison_table = pd.DataFrame({
    "Popularidade (Soma de Scores - Cenário A)": metrics_a_score_df["mean_score"],
    "Popularidade (Contagem - Cenário A)": metrics_a_count_df["mean_score"],
    "Popularidade (Soma de Scores - Cenário B)": metrics_b_score_df["mean_score"],
    "Popularidade (Contagem - Cenário B)": metrics_b_count_df["mean_score"],
    "Matrix Factorization (200 users)": pd.Series(mf_metrics)
})

print(f"=== Tabela Comparativa Geral (Top-{k} avaliado em {len(test_users)} usuários) ===")
comparison_table

=== Tabela Comparativa Geral (Top-10 avaliado em 17237 usuários) ===


,Popularidade (Soma de Scores - Cenário A),Popularidade (Contagem - Cenário A),Popularidade (Soma de Scores - Cenário B),Popularidade (Contagem - Cenário B),Matrix Factorization (200 users)
precision@10,0.001334,0.001398,0.001236,0.001329,0.002500
recall@10,0.006838,0.007709,0.006818,0.007674,0.018750
ndcg@10,0.003679,0.004603,0.003525,0.004458,0.010361
hit_rate@10,0.012647,0.013227,0.012009,0.012879,0.025000


# Log Metrics to MLflow

In [17]:
import mlflow

mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment(settings.mlflow_experiment_name)

# Vamos logar os resultados do Cenário A (Soma de Scores), que é o comparativo direto com a Fatoração de Matrizes
with mlflow.start_run(run_name="popularity_baseline"):
    # Log parâmetros do baseline
    mlflow.log_params({
        "k": k,
        "eval_users": len(test_users),
        "seed": settings.seed,
        "popularity_type": "score_sum"
    })
    
    # Log métricas obtidas (substituindo @ por _at_ para compatibilidade com o MLflow)
    for metric_name, row in metrics_a_score_df.iterrows():
        mlflow.log_metric(metric_name.replace("@", "_at_"), row["mean_score"])
        
    print("Métricas do baseline de popularidade logadas no MLflow com sucesso!")

Métricas do baseline de popularidade logadas no MLflow com sucesso!


# Final Comparison Summary

Analise o comportamento do baseline contra o modelo treinado:
- Para `_EVAL_USERS = 200`, o baseline de popularidade ficará zerado devido a amostragem e esparsidade do e-commerce, enquanto a Fatoração de Matrizes personalizada pontua em alguns poucos usuários.
- Para `_EVAL_USERS = len(ground_truth)` (amostra completa de teste), você verá as métricas globais reais do baseline de popularidade (ex: Hit Rate@10 de ~1.26%).